# Delivered Image Quality Trending (v1)

**Author:** Keith Bechtol, Aaron Roodman  
**Date Created:** 2026-05-04  
**Last Modified:** 2026-05-04  
**Status:** In Progress  
**Keywords:** image quality, PSF FWHM, ellipticity, AOS, donut blur, moment score, ConsDB, trending  

## Description

Trending of delivered image quality metrics for LSSTCam survey visits, queried from the ConsDB.

Key functionality:
1. Pull per-detector PSF moments and per-visit AOS / atmosphere metrics from ConsDB
2. Aggregate to per-visit and per-night summaries
3. Plot distributions and time-series of FWHM, ellipticity, and the moment score

**Output:** Both the parquet cache `ccdvisits_{day_obs_min}-{day_obs_max}.pq` and the PDF figures are written to the path in `output_dir` (default `blocks/output/`).

**Based on:** `image_quality_trending.ipynb` by Keith Bechtol. The single-shot 4-way ConsDB join was replaced with a chunked, explicit-join, dtype-aware fetch in `code/consdb_fetch.py` so multi-week date ranges load without OOM.

## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04 | Keith Bechtol | Original notebook |
| 2026-05-04 | Aaron Roodman | Adopted rubin-work template; swapped `fetch()` for chunked, explicit-join version in `code/consdb_fetch.py` to fix OOM on multi-week date ranges; removed `fetchA` / `fetchB` workaround |
| 2026-05-04 | Aaron Roodman | Cleanup: route parquet cache and PDF figures through `output_dir` (default `blocks/output/`); safer `no_proxy` mutation; tqdm progress bar in `fetch_chunked`; removed unused vars and dead commented-out plot blocks |


## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)


<a id='params'></a>
## Parameters

In [ ]:
# Parameters cell. Set defaults here

#day_obs_min = 20250620 # Start of SV surveys
#day_obs_max = 20250921 # Last night of SV surveys
#day_obs_min = 20251026 # Start of Early Operations
day_obs_min = 20260301 # March
#day_obs_min = 20260414 # Start of selection period
day_obs_max = 20260423 # End of selection period
#day_obs_max = 20260425

In [ ]:
from pathlib import Path

SAVE_FIGURES = True
SURVEY_PROGRAMS = ['BLOCK-365', 'BLOCK-407', 'BLOCK-408', 'BLOCK-416', 'BLOCK-417', 'BLOCK-419', 'BLOCK-421']  # Commissioning
#SURVEY_PROGRAMS = ['BLOCK-407', 'BLOCK-408', 'BLOCK-416', 'BLOCK-417', 'BLOCK-419', 'BLOCK-421']  # Early Ops
CAM_FWHM = 0.207  # arcsec

# Parquet cache lives in <topic>/output/ per rubin-work convention.
output_dir = Path('output')
output_dir.mkdir(parents=True, exist_ok=True)

# Figures: prefer output/, fall back to figures/, then cwd.
for _candidate in (Path('output'), Path('figures'), Path('.')):
    if _candidate.is_dir():
        figures_dir = _candidate
        break


<a id='setup'></a>
## Setup & Imports

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

#%matplotlib widget

In [ ]:
from lsst.utils.plotting import publication_plots
from lsst.utils.plotting import get_multiband_plot_colors

colors = get_multiband_plot_colors()
bands = colors.keys()

In [ ]:
publication_plots.set_rubin_plotstyle()

In [ ]:
from lsst.summit.utils import (
    getAirmassSeeingCorrection,
    getBandpassSeeingCorrection,
)

In [ ]:
import os
os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"

from lsst.summit.utils import ConsDbClient

client = ConsDbClient("http://consdb-pq.consdb:8080/consdb")


In [ ]:
instrument = 'lsstcam'

<a id='functions'></a>
## Helper Functions

In [ ]:
import sys
from pathlib import Path

# Make code/ importable from this notebook
sys.path.insert(0, str(Path.cwd() / 'code'))
from consdb_fetch import fetch_chunked

In [ ]:
from astropy.time import Time


def dayObsToTime(day_obs):
    """Convert a YYYYMMDD day_obs integer to an astropy Time at noon UTC."""
    s = str(int(day_obs))
    return Time(f'{s[0:4]}-{s[4:6]}-{s[6:8]} 12:00:00', format='iso')


<a id='data'></a>
## Data Access

In [ ]:
ccdvisits = fetch_chunked(
    client, instrument, day_obs_min, day_obs_max,
    SURVEY_PROGRAMS, chunk_days=7,
)
print(len(ccdvisits))

<a id='analysis'></a>
## Analysis

In [ ]:
airmass_seeing_correction = np.array([getAirmassSeeingCorrection(airmass) for airmass in ccdvisits["airmass"]])
bandpass_seeing_correction = np.array([getBandpassSeeingCorrection(band) for band in ccdvisits["physical_filter"]])
ccdvisits["psf_fwhm_zenith_500nm"] = ccdvisits["psf_fwhm"] * airmass_seeing_correction * bandpass_seeing_correction

ccdvisits['donut_blur_atm_fwhm'] = np.sqrt(ccdvisits['donut_blur_fwhm']**2 - CAM_FWHM**2)
ccdvisits['aos_cam_fwhm'] = np.sqrt(ccdvisits['aos_fwhm']**2 + CAM_FWHM**2)

ccdvisits['coma'] = np.sqrt(ccdvisits['coma_1']**2 + ccdvisits['coma_2']**2)
ccdvisits['trefoil'] = np.sqrt(ccdvisits['trefoil_1']**2 + ccdvisits['trefoil_2']**2)
ccdvisits['moment_score'] = 3 * ccdvisits['coma']**2 + ccdvisits['trefoil']**2

In [ ]:
outfile = output_dir / f'ccdvisits_{day_obs_min}-{day_obs_max}.pq'
print(outfile)


In [ ]:
ccdvisits.to_parquet(outfile)

In [ ]:
ccdvisits_read = pd.read_parquet(outfile)
assert np.all(ccdvisits['visit_id'] == ccdvisits_read['visit_id'])
assert np.all(ccdvisits['detector'] == ccdvisits_read['detector'])
assert np.all(ccdvisits.columns == ccdvisits_read.columns)

In [ ]:
ccdvisits = pd.read_parquet(outfile)

### Per-visit summary table

In [ ]:
groups = ccdvisits.groupby('visit_id')
visits_summary = pd.DataFrame({
    'day_obs': groups['day_obs'].first(),
    'target_name': groups['target_name'].first(),
    'science_program': groups['science_program'].first(),
    'observation_reason': groups['observation_reason'].first(),
    'seq_num': groups['seq_num'].median(),
    'exp_midpt_mjd': groups['exp_midpt_mjd'].median(),
    'donut_blur_fwhm': groups['donut_blur_fwhm'].median(),
    'aos_fwhm': groups['aos_fwhm'].median(),
    'donut_blur_atm_fwhm': groups['donut_blur_atm_fwhm'].median(),
    'aos_cam_fwhm': groups['aos_cam_fwhm'].median(),
    'physical_rotator_angle': groups['physical_rotator_angle'].median(),
    'altitude': groups['altitude'].median(),
    'airmass': groups['airmass'].median(),
    'psf_fwhm_05': groups['psf_fwhm'].quantile(0.05),
    'psf_fwhm_50': groups['psf_fwhm'].quantile(0.50),
    'psf_fwhm_95': groups['psf_fwhm'].quantile(0.95),
    'psf_fwhm_zenith_500nm_50': groups['psf_fwhm_zenith_500nm'].quantile(0.50),
    'psf_fwhm_area_50': groups['psf_fwhm_area'].quantile(0.50),
    'psf_e_05': groups['psf_e'].quantile(0.05),
    'psf_e_50': groups['psf_e'].quantile(0.50),
    'psf_e_95': groups['psf_e'].quantile(0.95),
    'psf_e1_05': groups['psf_e1'].quantile(0.05),
    'psf_e1_50': groups['psf_e1'].quantile(0.50),
    'psf_e1_95': groups['psf_e1'].quantile(0.95),
    'psf_e2_05': groups['psf_e2'].quantile(0.05),
    'psf_e2_50': groups['psf_e2'].quantile(0.50),
    'psf_e2_95': groups['psf_e2'].quantile(0.95),
    #'psf_e_unnormalized_05': groups['psf_e_unnormalized'].quantile(0.05),
    #'psf_e_unnormalized_50': groups['psf_e_unnormalized'].quantile(0.50),
    #'psf_e_unnormalized_95': groups['psf_e_unnormalized'].quantile(0.95),
    'band': groups['band'].first(),
})
visits_summary['psf_fwhm_95_05'] = np.sqrt(visits_summary['psf_fwhm_95']**2 - visits_summary['psf_fwhm_05']**2)
visits_summary['sys_50'] = np.sqrt(visits_summary['psf_fwhm_50']**2 + CAM_FWHM**2 - visits_summary['donut_blur_fwhm']**2)
visits_summary['psf_fwhm_model'] = np.sqrt(visits_summary['aos_fwhm']**2 + visits_summary['donut_blur_fwhm']**2)

### Per-night summary table

In [ ]:
selection = (visits_summary['band'] != "y") & ~visits_summary['observation_reason'].str.contains('filter_change_close_loop')
selection = selection & ~((visits_summary['day_obs'] == 20260424) & (visits_summary['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation

groups = visits_summary[selection].groupby('day_obs')
day_obs_summary = pd.DataFrame({
    'day_obs': groups['day_obs'].first(),
    'n_visits': groups['day_obs'].count(),
    'psf_fwhm_95_05_low': groups['psf_fwhm_95_05'].quantile(0.10),
    'psf_fwhm_95_05_50': groups['psf_fwhm_95_05'].quantile(0.50),
    'psf_fwhm_95_05_high': groups['psf_fwhm_95_05'].quantile(0.90),
    'aos_fwhm_low': groups['aos_fwhm'].quantile(0.10),
    'aos_fwhm_50': groups['aos_fwhm'].quantile(0.50),
    'aos_fwhm_high': groups['aos_fwhm'].quantile(0.90),
    'aos_cam_fwhm_low': groups['aos_cam_fwhm'].quantile(0.10),
    'aos_cam_fwhm_50': groups['aos_cam_fwhm'].quantile(0.50),
    'aos_cam_fwhm_high': groups['aos_cam_fwhm'].quantile(0.90),
    'sys_50_low': groups['sys_50'].quantile(0.10),
    'sys_50_50': groups['sys_50'].quantile(0.50),
    'sys_50_high': groups['sys_50'].quantile(0.90),
    'psf_e_50_low': groups['psf_e_50'].quantile(0.10),
    'psf_e_50_50': groups['psf_e_50'].quantile(0.50),
    'psf_e_50_high': groups['psf_e_50'].quantile(0.90),
    'psf_e1_50_low': groups['psf_e1_50'].quantile(0.10),
    'psf_e1_50_50': groups['psf_e1_50'].quantile(0.50),
    'psf_e1_50_high': groups['psf_e1_50'].quantile(0.90),
    'psf_e2_50_low': groups['psf_e2_50'].quantile(0.10),
    'psf_e2_50_50': groups['psf_e2_50'].quantile(0.50),
    'psf_e2_50_high': groups['psf_e2_50'].quantile(0.90),
    'psf_fwhm_50_low': groups['psf_fwhm_50'].quantile(0.10),
    'psf_fwhm_50_50': groups['psf_fwhm_50'].quantile(0.50),
    'psf_fwhm_50_high': groups['psf_fwhm_50'].quantile(0.90),
    'donut_blur_atm_fwhm_low': groups['donut_blur_atm_fwhm'].quantile(0.10),
    'donut_blur_atm_fwhm_50': groups['donut_blur_atm_fwhm'].quantile(0.50),
    'donut_blur_atm_fwhm_high': groups['donut_blur_atm_fwhm'].quantile(0.90),
})

<a id='results'></a>
## Results & Plots

### Per-detector distributions

In [ ]:
#selection = (ccdvisits['day_obs'] >= 20251130) & (ccdvisits['day_obs'] <= 20251202)
selection = (ccdvisits['day_obs'] >= day_obs_min) & (ccdvisits['day_obs'] <= day_obs_max)
#print(np.sum(selection) / 189.)


bins = np.linspace(-0.4, 0.4, 81)

plt.figure()
plt.hist(ccdvisits['psf_e'][selection], bins=bins, density=True, histtype='step', lw=2)# label='PSF e')
#plt.hist(ccdvisits['psf_e1'][selection], bins=bins, density=True, histtype='step', lw=2, label='PSF e1')
#plt.hist(ccdvisits['psf_e2'][selection], bins=bins, density=True, histtype='step', lw=2, label='PSF e2')

plt.axvline(np.nanmedian(ccdvisits['psf_e'][selection]), ls='--', lw=1, c='0.5', label='Median = %.2f'%(np.nanmedian(ccdvisits['psf_e'][selection])))
print(np.nanmedian(ccdvisits['psf_e'][selection]))

plt.legend()
plt.xlabel('Ellipticity')
plt.ylabel('PDF')
#plt.xlim(bins[0], bins[-1])
plt.xlim(0, bins[-1])
plt.ylim(0., plt.ylim()[-1])
plt.title(f"{min(ccdvisits['day_obs'][selection])} - {max(ccdvisits['day_obs'][selection])}")
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'ellipticity_hist_{day_obs_min}-{day_obs_max}.pdf')
plt.show()

In [ ]:
bins = np.linspace(0.5, 2., 51)

plt.figure()

for band in bands:
    selection = (ccdvisits["band"] == band)
    plt.hist(ccdvisits["psf_fwhm"][selection], bins=bins, color=colors[band], density=True, histtype="step", lw=2, label=band)

plt.grid()
plt.xlabel('PSF FWHM (arcsec)')
plt.ylabel('Fraction of Sensors')
plt.legend()
plt.xlim(np.min(bins), np.max(bins))
plt.title(f'{day_obs_min}-{day_obs_max}')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'psf_fwhm_hist_{day_obs_min}-{day_obs_max}.pdf')
plt.show()

In [ ]:
bins = np.linspace(0.5, 2.5, 51)

plt.figure()

for band in bands:
    selection = (ccdvisits["band"] == band)
    plt.hist(ccdvisits["psf_fwhm"][selection], bins=bins, color=colors[band], density=True, histtype="step", cumulative=True, lw=2, label=band)
plt.grid()
plt.xlabel('PSF FWHM (arcsec)')
plt.ylabel('Fraction of Sensors')
plt.legend(loc='lower right')
plt.xlim(np.min(bins), np.max(bins))
plt.ylim(0., 1.)
plt.title(f'{day_obs_min}-{day_obs_max}')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'psf_fwhm_cumulative_{day_obs_min}-{day_obs_max}.pdf')
plt.show()

In [ ]:
log = True
bins = np.linspace(0, 0.2, 1001)

selection = np.isfinite(ccdvisits['moment_score'])
selection = selection & (ccdvisits['band'] != 'y') # Exclude y filter
selection = selection & ~((ccdvisits['day_obs'] == 20260424) & (ccdvisits['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation

plt.figure()
plt.hist(ccdvisits['moment_score'][selection], bins=bins, cumulative=-1, density=True, histtype='step', color='black')

thresholds = 4. * np.array([0.002, 0.005, 0.02])
for threshold in thresholds:
    plt.axvline(threshold, color='0.5', ls='--', lw=1)

    fraction = np.sum(
        (ccdvisits['moment_score'][selection] > threshold) / float(len(ccdvisits['moment_score'][selection]))
    )
    plt.axhline(fraction, color='0.5', ls='--', lw=1)

plt.xlim(bins[0], 0.2)

plt.axvspan(0., 4. * 0.002, color='tab:blue', alpha=0.1)
plt.axvspan(4. * 0.002, 4. * 0.005, color='tab:green', alpha=0.1)
plt.axvspan(4. * 0.005, 4. * 0.02, color='yellow', alpha=0.1)
plt.axvspan(4. * 0.02, 4. * 0.05, color='tab:red', alpha=0.1)

plt.text(0.008 - 0.001, 0.9, 'Excellent', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
plt.text(0.02 - 0.001, 0.9, 'Good', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
plt.text(0.08 - 0.001, 0.9, 'Marginal', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
plt.text(0.08 + 0.001, 0.9, 'Poor', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='left', color='0.5')

if log:
    plt.ylim(0.0001, 1.)
    plt.yscale('log')
else:
    plt.ylim(0., 1.)
plt.xlabel('Moment Score')
plt.ylabel('1 - CDF')
plt.title(f'{day_obs_min} - {day_obs_max}')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'moment_score_cumulative_{day_obs_min}-{day_obs_max}.pdf')
plt.show()

In [ ]:
log = True
bins = np.linspace(0, 0.2, 1001)

selection_donut_blur = (ccdvisits['donut_blur_fwhm'] < 0.8)
selection = np.isfinite(ccdvisits['moment_score'])
selection = selection & (ccdvisits['band'] != 'y') # Exclude y filter
selection = selection & ~((ccdvisits['day_obs'] == 20260424) & (ccdvisits['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation

plt.figure()
plt.hist(ccdvisits['moment_score'][selection_donut_blur & selection], bins=bins, cumulative=-1, density=True, histtype='step', color='black', label='Donut Blur < 0.8')
plt.hist(ccdvisits['moment_score'][~selection_donut_blur & selection], bins=bins, cumulative=-1, density=True, histtype='step', color='black', ls='--', label='Donut Blur > 0.8')

thresholds = 4. * np.array([0.002, 0.005, 0.02])
for threshold in thresholds:
    plt.axvline(threshold, color='0.5', ls='--', lw=1)

    #fraction = np.sum(ccdvisits['moment_score'] > threshold) / float(len(ccdvisits['moment_score']))
    #plt.axhline(fraction, color='0.5', ls='--', lw=1)

plt.xlim(bins[0], 0.2)

plt.axvspan(0., 4. * 0.002, color='tab:blue', alpha=0.1)
plt.axvspan(4. * 0.002, 4. * 0.005, color='tab:green', alpha=0.1)
plt.axvspan(4. * 0.005, 4. * 0.02, color='yellow', alpha=0.1)
plt.axvspan(4. * 0.02, 4. * 0.05, color='tab:red', alpha=0.1)

plt.text(0.008 - 0.001, 0.9, 'Excellent', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
plt.text(0.02 - 0.001, 0.9, 'Good', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
plt.text(0.08 - 0.001, 0.9, 'Marginal', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='right', color='0.5')
plt.text(0.08 + 0.001, 0.9, 'Poor', rotation=90, fontsize=8, verticalalignment='top', horizontalalignment='left', color='0.5')

if log:
    plt.ylim(0.0001, 1.)
    plt.yscale('log')
else:
    plt.ylim(0., 1.)
plt.legend()
plt.xlabel('Moment Score')
plt.ylabel('1 - CDF')
plt.title(f'{day_obs_min} - {day_obs_max}')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'moment_score_cumulative_donut_blur_{day_obs_min}-{day_obs_max}.pdf')
plt.show()

### Per-visit distributions

In [ ]:
selection = (visits_summary['band'] != "y") & ~visits_summary['observation_reason'].str.contains('filter_change_close_loop')
selection = selection & ~((visits_summary['day_obs'] == 20260424) & (visits_summary['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation

bins = np.linspace(0., 1.2, 61)

plt.figure()

plt.hist(visits_summary['psf_fwhm_95_05'][selection], bins=bins, histtype='step', density=True, lw=2, color='tab:blue', label='Variation across FoV')
plt.hist(visits_summary['sys_50'][selection], bins=bins, histtype='step', density=True, lw=2, color='tab:red', label='Delivered - Estimated Atmosphere')
plt.hist(visits_summary['aos_cam_fwhm'][selection], bins=bins, histtype='step', density=True, lw=2, color='tab:purple', label='Optics + Camera Diffusion')
plt.axvspan(0., 0.45, color='tab:green', alpha=0.1)
plt.xlim(0., 1.2)

print(np.nanmedian(visits_summary['psf_fwhm_95_05'][selection]))
print(np.nanmedian(visits_summary['sys_50'][selection]))
print(np.nanmedian(visits_summary['aos_cam_fwhm'][selection]))

plt.axvline(np.nanmedian(visits_summary['psf_fwhm_95_05'][selection]), ls='--', lw=1, c='tab:blue')
plt.axvline(np.nanmedian(visits_summary['sys_50'][selection]), ls='--', lw=1, c='tab:red')
plt.axvline(np.nanmedian(visits_summary['aos_cam_fwhm'][selection]), ls='--', lw=1, c='tab:purple')

plt.xlabel('Estimated Instrument Contribution (arcsec)')
plt.ylabel('PDF')
plt.legend(loc='upper right')
plt.title(f'{day_obs_min} - {day_obs_max}')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'diq_instrument_contribution_hist_{day_obs_min}-{day_obs_max}.pdf')
plt.show()

### Trending

In [ ]:
selection = np.tile(True, len(day_obs_summary))

#xticks = np.arange(len(day_obs_summary[selection]))
xticks = np.array([dayObsToTime(_).mjd for _ in day_obs_summary['day_obs'][selection]])

fig, ax = plt.subplots(figsize=(15, 6), dpi=200)

variation = ax.errorbar(
    xticks - 0.15, day_obs_summary['psf_fwhm_95_05_50'][selection],
    yerr=np.array([
        day_obs_summary['psf_fwhm_95_05_50'][selection] - day_obs_summary['psf_fwhm_95_05_low'][selection], 
        day_obs_summary['psf_fwhm_95_05_high'][selection] - day_obs_summary['psf_fwhm_95_05_50'][selection]
    ]), 
    fmt='_', c='tab:blue', label='Variation Across FoV'
    #label='sqrt(fwhm_95^2 - fwhm_5^2)'
)
aos = ax.errorbar(
    xticks + 0., day_obs_summary['aos_cam_fwhm_50'][selection],
    yerr=np.array([day_obs_summary['aos_cam_fwhm_50'][selection] - day_obs_summary['aos_cam_fwhm_low'][selection], 
                   day_obs_summary['aos_cam_fwhm_high'][selection] - day_obs_summary['aos_cam_fwhm_50'][selection]]), 
    fmt='_', c='tab:purple', label='Optics + Camera Diffusion'
    #label='sqrt(aos_fwhm^2 + cam_fwhm^2)'
)
sys = ax.errorbar(
    xticks + 0.15, day_obs_summary['sys_50_50'][selection],
    yerr=np.array([day_obs_summary['sys_50_50'][selection] - day_obs_summary['sys_50_low'][selection], 
                   day_obs_summary['sys_50_high'][selection] - day_obs_summary['sys_50_50'][selection]]), 
    fmt='_', c='tab:red', label='Delivered - Estimated Atmosphere'
    #label='sqrt(fwhm_50^2 - (donut_blur^2 - cam_fwhm^2))'
)


ellipticity = ax.errorbar(
    xticks + 0., day_obs_summary['psf_e_50_50'][selection],
    yerr=np.array([day_obs_summary['psf_e_50_50'][selection] - day_obs_summary['psf_e_50_low'][selection], 
                   day_obs_summary['psf_e_50_high'][selection] - day_obs_summary['psf_e_50_50'][selection]]), 
    fmt='_', c='tab:orange', label='PSF Ellipticity'
    #label='psf_e'
)


first_legend = ax.legend(handles=[variation, aos, sys], loc='upper left', title='Estimated Instrument Contribution')
#first_legend = ax.legend(handles=[variation, aos], loc='upper left', title='Estimated System Contribution')
#second_legend = ax.legend(handles=[psf, atm, ellipticity], loc='upper right', title='Measured Quantities')
second_legend = ax.legend(handles=[ellipticity], loc='upper right', title='Measured Quantities')
ax.add_artist(first_legend)

ax.set_xticks(xticks, day_obs_summary['day_obs'][selection], rotation=90, fontsize=7.5)
#ax.set_ylim(0, plt.ylim()[-1])
#ax.set_ylim(0, 2.5)
ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
ax.set_ylim(0., 1.0)
#ax.set_ylim(0.3, 0.5)
#for y in [0.04, 0.2, 0.4, 0.6]:
#    plt.axhline(y, c='0.75', ls='--', zorder=0)

#ax.axhline(1., c='0.25', ls=':', zorder=0)
ax.axhline(0.45, c='tab:green', ls=':', zorder=0)
ax.axhline(0.04, c='tab:orange', ls=':', zorder=0)

plt.axhspan(0.04, 0.45, color='tab:green', alpha=0.1)
plt.axhspan(0., 0.04, color='tab:orange', alpha=0.1)

ax.set_ylabel('Image Quality Metric (arcsec)')
plt.title('10th, 50th, 90th Percentiles of Per-visit Quantities for Each Night')
plt.grid()
plt.minorticks_off()
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'diq_instrument_contribution_timeseries_{day_obs_min}-{day_obs_max}.pdf')
fig.tight_layout()
plt.show()

In [ ]:
selection = np.tile(True, len(day_obs_summary))

#xticks = np.arange(len(day_obs_summary[selection]))
xticks = np.array([dayObsToTime(_).mjd for _ in day_obs_summary['day_obs'][selection]])

fig, ax = plt.subplots(figsize=(15, 6), dpi=200)
psf = ax.errorbar(
    xticks - 0.075, day_obs_summary['psf_fwhm_50_50'][selection],
    yerr=np.array([day_obs_summary['psf_fwhm_50_50'][selection] - day_obs_summary['psf_fwhm_50_low'][selection], 
                   day_obs_summary['psf_fwhm_50_high'][selection] - day_obs_summary['psf_fwhm_50_50'][selection]]), 
    fmt='_', c='black', label='Delivered Median PSF FWHM' 
    #label='fwhm_50'
)
atm = ax.errorbar(
    xticks + 0.075, day_obs_summary['donut_blur_atm_fwhm_50'][selection],
    yerr=np.array([day_obs_summary['donut_blur_atm_fwhm_50'][selection] - day_obs_summary['donut_blur_atm_fwhm_low'][selection], 
                   day_obs_summary['donut_blur_atm_fwhm_high'][selection] - day_obs_summary['donut_blur_atm_fwhm_50'][selection]]), 
    fmt='_', c='0.5', label='Estimated Atmosphere'
    #label='sqrt(donut_blur^2 - cam_fwhm^2)'
)



#first_legend = ax.legend(handles=[variation, aos, sys], loc='upper left', title='Estimated System Contribution')
#first_legend = ax.legend(handles=[variation, aos], loc='upper left', title='Estimated System Contribution')
#second_legend = ax.legend(handles=[psf, atm, ellipticity], loc='upper right', title='Measured Quantities')
second_legend = ax.legend(handles=[psf, atm], loc='upper right', title='Measured Quantities')
ax.add_artist(second_legend)

ax.set_xticks(xticks, day_obs_summary['day_obs'][selection], rotation=90, fontsize=7.5)
ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
#ax.set_ylim(0, plt.ylim()[-1])
#ax.set_ylim(0, 2.5)
ax.set_ylim(0., 2.0)
#ax.set_ylim(0.3, 0.5)
#for y in [0.04, 0.2, 0.4, 0.6]:
#    plt.axhline(y, c='0.75', ls='--', zorder=0)

#ax.axhline(1., c='0.25', ls=':', zorder=0)

ax.set_ylabel('Image Quality Metric (arcsec)')
plt.title('10th, 50th, 90th Percentiles of Per-visit Quantities for Each Night')
plt.grid()
plt.minorticks_off()
fig.tight_layout()
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'diq_measured_timeseries_{day_obs_min}-{day_obs_max}.pdf')
plt.show()


In [ ]:
APPLY_DONUT_BLUR_SELECTION = False
DONUT_BLUR_THRESHOLD = 0.8 # arcsec

if APPLY_DONUT_BLUR_SELECTION:
    selection_donut_blur = (ccdvisits['donut_blur_fwhm'] < DONUT_BLUR_THRESHOLD)
else:
    selection_donut_blur = np.tile(True, len(ccdvisits))

selection_donut_blur = selection_donut_blur & (ccdvisits['band'] != 'y') # Exclude y filter
selection_donut_blur = selection_donut_blur & ~((ccdvisits['day_obs'] == 20260424) & (ccdvisits['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation

groups = ccdvisits[selection_donut_blur].groupby('day_obs')
day_obs_summary_moments = pd.DataFrame({
    'day_obs': groups['day_obs'].first(),
    'n_ccdvisits': groups['day_obs'].count(),
    'moment_score_low': groups['moment_score'].agg(lambda x: (x <= 4. * 0.002).mean()),
    'moment_score_002': groups['moment_score'].agg(lambda x: (x > 4. * 0.002).mean()),
    'moment_score_005': groups['moment_score'].agg(lambda x: (x > 4. * 0.005).mean()),
    'moment_score_02': groups['moment_score'].agg(lambda x: (x > 4. * 0.02).mean()),
})


selection = np.tile(True, len(day_obs_summary_moments))
xticks = np.array([dayObsToTime(_).mjd for _ in day_obs_summary_moments['day_obs'][selection]])

fig, ax = plt.subplots(figsize=(15, 6), dpi=200)

ax.scatter(xticks + 0., day_obs_summary_moments['moment_score_low'][selection], c='tab:blue', label='Excellent (Moment Score < 0.008)')
ax.scatter(xticks + 0., 1. - day_obs_summary_moments['moment_score_005'][selection], c='tab:green', label='Excellent + Good (Moment Score < 0.02)')

ax.legend(loc='lower right')

ax.set_xticks(xticks, day_obs_summary_moments['day_obs'][selection], rotation=90, fontsize=7.5)
ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
ax.set_ylim(0., 1.0)
ax.set_ylabel('Fraction of Detector Images\nMeeting Criteria')
plt.grid()
plt.minorticks_off()
if APPLY_DONUT_BLUR_SELECTION:
    plt.title(f'Donut Blur < {DONUT_BLUR_THRESHOLD}')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'moment_score_timeseries_{day_obs_min}-{day_obs_max}.pdf')
plt.show()

In [ ]:
APPLY_DONUT_BLUR_SELECTION = True
DONUT_BLUR_THRESHOLD = 0.8 # arcsec

if APPLY_DONUT_BLUR_SELECTION:
    selection_donut_blur = (ccdvisits['donut_blur_fwhm'] < DONUT_BLUR_THRESHOLD)
else:
    selection_donut_blur = np.tile(True, len(ccdvisits))

selection_donut_blur = selection_donut_blur & (ccdvisits['band'] != 'y') # Exclude y filter
selection_donut_blur = selection_donut_blur & ~((ccdvisits['day_obs'] == 20260424) & (ccdvisits['seq_num'] < 398)) # Remove corrupted period due to lack of hexapod compensation

groups = ccdvisits[selection_donut_blur].groupby('day_obs')
day_obs_summary_moments = pd.DataFrame({
    'day_obs': groups['day_obs'].first(),
    'n_ccdvisits': groups['day_obs'].count(),
    'moment_score_low': groups['moment_score'].agg(lambda x: (x <= 4. * 0.002).mean()),
    'moment_score_002': groups['moment_score'].agg(lambda x: (x > 4. * 0.002).mean()),
    'moment_score_005': groups['moment_score'].agg(lambda x: (x > 4. * 0.005).mean()),
    'moment_score_02': groups['moment_score'].agg(lambda x: (x > 4. * 0.02).mean()),
})


selection = np.tile(True, len(day_obs_summary_moments))
xticks = np.array([dayObsToTime(_).mjd for _ in day_obs_summary_moments['day_obs'][selection]])

fig, ax = plt.subplots(figsize=(15, 6), dpi=200)

ax.scatter(xticks + 0., day_obs_summary_moments['moment_score_low'][selection], c='tab:blue', label='LOW (Moment Score < 0.008)')
ax.scatter(xticks + 0., 1. - day_obs_summary_moments['moment_score_005'][selection], c='tab:green', label='LOW + MEDIUM (Moment Score < 0.02)')

ax.legend(loc='lower right')

ax.set_xticks(xticks, day_obs_summary_moments['day_obs'][selection], rotation=90, fontsize=7.5)
ax.set_xlim(xticks[0] - 1, xticks[-1] + 1)
ax.set_ylim(0., 1.0)
ax.set_ylabel('Fraction of Detector Images\nMeeting Criteria')
plt.grid()
plt.minorticks_off()
if APPLY_DONUT_BLUR_SELECTION:
    plt.title(f'Donut Blur < {DONUT_BLUR_THRESHOLD}')
plt.tight_layout()
if SAVE_FIGURES:
    plt.savefig(figures_dir / f'moment_score_timeseries_donut_blur_{day_obs_min}-{day_obs_max}.pdf')
plt.show()